In [ ]:
!pip install -q pymorphy3

# Импорты

In [ ]:
!pip install -q papermill

import papermill as pm
import pandas as pd
import gdown
import os

In [ ]:
if 'input_filename' not in locals():
    input_filename = '/content/data_1_2_well_cleanedv2.xlsx' # имя файла по умолчанию для ручного запуска
    print(f"Запуск вручную. Используем файл: {input_filename}")
else:
    print(f"Пайплайн запущен! Обрабатываем файл: {input_filename}")

if os.path.exists(input_filename):
    df = pd.read_excel(input_filename)
    print("Файл успешно загружен в DataFrame")
else:
    print(f"Ошибка! Файла {input_filename} нет в папке /content/")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import requests
import pymorphy3
import re
import joblib

In [ ]:
url = "https://raw.githubusercontent.com/stopwords-iso/stopwords-ru/refs/heads/master/stopwords-ru.txt"
req = requests.get(url)
stop_words = req.text.split("\n")

In [ ]:
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
#df = pd.read_excel("/content/data_1_2_well_cleanedv2.xlsx")
df.head(1)

# Предобработка

In [ ]:
df_comment = df[['comments', 'id']]

In [ ]:
df_comment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11013 entries, 0 to 11012
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   comments  2112 non-null   object
 1   id        11013 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 172.2+ KB


In [ ]:
df_comment = df_comment.fillna('')

In [ ]:
df_comment['Длина комментария'] = df_comment['comments'].str.split().str.len()

# Предобработка текста

In [ ]:
morph = pymorphy3.MorphAnalyzer()
cache = {}

def clean_text_full(text):
    if not isinstance(text, str) or not text.strip():
        return ""

    #  только буквы
    text = re.sub(r'[^а-яА-ЯёЁ ]', ' ', text.lower())
    words = text.split()

    result = []
    for word in words:
        if len(word) > 1 or word == 'я':
            if word not in cache:
                cache[word] = morph.parse(word)[0].normal_form
            result.append(cache[word])

    return " ".join(result)

df_comment['comment'] = df_comment['comments'].apply(clean_text_full)
df_comment['comment'] = df_comment['comment'].fillna('').astype(str)

In [ ]:
df_comment['comment'] = df_comment['comment'].astype(str)

In [ ]:
df_comment = df_comment.drop(columns='comments')

In [ ]:
df_comment['Длина комментария'] = df_comment['Длина комментария'].fillna(0)

In [ ]:
df_comment = df_comment.fillna('')
df_comment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11013 entries, 0 to 11012
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 11013 non-null  int64  
 1   Длина комментария  11013 non-null  float64
 2   comment            11013 non-null  object 
dtypes: float64(1), int64(1), object(1)
memory usage: 258.2+ KB


# Применение модели

In [ ]:
target_names = ['Зарплата', 'Дефицит', 'Возможности для развития', 'Рабочие процессы',
       'Условия работы', 'График работы', 'Социальный пакет',
       'Нет комментариев', 'Комментарии к опроснику',
       'Благодарность за опросник', 'Скептицизм к изменениям',
       'Надежда на изменения', 'Позитив-неуточненно',
       'Руководство, коллеги и репутация организации',
       'Взаимодействие с пациентами, проф риски, востребованность',
       'Инфраструктура и здравоохранение региона']

loaded_pipeline = joblib.load('LinearSVC_classifier_model_11may.pkl')

predictions = loaded_pipeline.predict(df_comment)

tags_list = [
    [target_names[i] for i, value in enumerate(row) if value == 1]
    for row in predictions
]

df_results = df_comment.copy()
df_results['predicted_comment_tags'] = tags_list

df_results.head(10)

In [ ]:
df = df.merge(df_results, on='id', how='left')
df.head(2)

# Экспорт файла

In [ ]:
output_filename = 'with_comment_tags'+input_filename
print(output_filename)

preprocessing_result_11_05_2026.xlsx


In [ ]:
df.to_excel(output_filename, index=False)

print(f"Файл успешно сохранен как: {output_filename}")